In [ ]:
import kagglehub
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
Q1_data_path = os.path.join(path, 'Q1_data.csv')
df_Q1_data = pd.read_csv(Q1_data_path)


In [ ]:
print(f"Shape: {df_Q1_data.shape}")
df_Q1_data.head()

In [ ]:
df_Q1_data.info()

In [ ]:
df_Q1_data.describe()

In [ ]:
print(f"Delivery Time more than 50: {(df_Q1_data['Delivery_Time'] > 50.0).sum()}")
print(f"Delivery Time less than 50: {(df_Q1_data['Delivery_Time'] < 50.0).sum()}")

In [ ]:
print(f"Low: {(df_Q1_data['Traffic_Level'] =="Low").sum()}")
print(f"Medium: {(df_Q1_data['Traffic_Level'] == "Medium").sum()}")

In [ ]:

stat_cols = ['Order_ID']
df_clean = df_Q1_data.dropna(subset=stat_cols).copy()
print(f"Shape after cleaning: {df_clean.shape}")


In [ ]:
print("Missing values:")
print(df_Q1_data.isnull().sum())
for col in ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type', 'Preparation_Time_min','Courier_Experience_yrs', 'Delivery_Time' ]:
    df_clean[col] = df_clean[col].fillna('unknown')

In [ ]:
# Define features and target
feature_cols = col
X = df_clean[feature_cols]
y = df_clean['Weather']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Encode type columns
le = LabelEncoder()
df_clean['Weather'] = le.fit_transform(df_clean['Weather'])

In [ ]:
# Task 5: Write your code here:

In [ ]:
# Task 6: Write your code here:

In [ ]:
# Separate Features and Target
target_column = "Delivery_Time"

X = df_Q1_data.drop(target_column, axis=1)
y = df_Q1_data[target_column]

print("\nDataset Shapes")
print("X:", X.shape)
print("y:", y.shape)

In [ ]:
# Encode Categorical Features
label_encoder = LabelEncoder()

for col in X.select_dtypes(include=["object"]).columns:
    X[col] = label_encoder.fit_transform(X[col])

# Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Define Model
model = LinearRegression()

# K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mse_scores = []
mae_scores = []

for train_idx, test_idx in kf.split(X_scaled):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Train model
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Evaluation metrics
    mse_scores.append(mean_squared_error(y_test, y_pred))
    mae_scores.append(mean_absolute_error(y_test, y_pred))


# Print Evaluation Metrics
print("\nModel Evaluation Metrics (K-Fold)\n" + "-"*40)
print(f"MSE : {np.mean(mse_scores):.2f}")
print(f"MAE : {np.mean(mae_scores):.2f}")
print(f"RMSE: {np.sqrt(np.mean(mse_scores)):.2f}")
print("-"*40)

In [ ]:
# Plot Predictions vs Ground Truth
for train_idx, test_idx in kf.split(X_scaled):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Train model
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    "r--",
    linewidth=2
)

plt.xlabel("Actual Delivery_Time")
plt.ylabel("Predicted Delivery_Time")
plt.title("Linear Regression: Predictions vs Ground Truth")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task Bonus: Write your code here: